# Egy nap részletesen + trendek
Állítsd be a `DAY`-t (vagy hagyd `None`-on az utolsó teljes naphoz), majd **Run All**.

Forrás: az `intraday`, `training_readiness`, `training_status`, `daily_summary`, `sleep`, `hrv` táblák (#6).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from mygarmin.analysis import load_day, daily_trends, available_days, STRESS_LABELS, TZ

DATA = Path.cwd().parent / "data" if Path.cwd().name == "notebooks" else Path.cwd() / "data"
DB = DATA / "garmin.db"
print("Elérhető napok:", ", ".join(available_days(DB)))

In [ ]:
DAY = None  # pl. "2026-09-20"; None = utolsó teljes nap (tegnap)

day = load_day(DB, DAY)
s = day.summary
morning = day.readiness[day.readiness["is_morning"] == 1].head(1)


def hm(sec):
    return f"{int(sec // 3600)}:{int(sec % 3600 // 60):02d}" if sec and sec == sec else None


pd.Series({
    "Nap": day.date,
    "Lépés": s.get("steps"),
    "Nyugalmi / min / max pulzus": f"{s.get('resting_hr')} / {s.get('min_hr')} / {s.get('max_hr')}",
    "Átlag / max stressz": f"{s.get('avg_stress')} / {s.get('max_stress')}",
    "Body Battery max / min": f"{s.get('body_battery_high')} / {s.get('body_battery_low')}",
    "Alvás (ó:p) / pontszám": f"{hm(s.get('sleep_s'))} / {s.get('sleep_score')}",
    "Éjszakai HRV (ms)": s.get("hrv"),
    "Reggeli edzéskészség": None if morning.empty else f"{morning.iloc[0]['score']} ({morning.iloc[0]['level']})",
    "Aktivitások": ", ".join(day.activities["name"]) or "-",
}, name="érték").to_frame()

## A nap idővonala
Kék sáv = alvás, narancs sáv = aktivitás. A stressz aktivitás közben nem mérhető (üres), a Body Battery alvás alatt töltődik, stresszes / aktív időben merül.

In [ ]:
STRESS_COLORS = {"pihenés": "#4a90d9", "alacsony": "#f5c26b", "közepes": "#f08c3a", "magas": "#d9534f"}
t0 = pd.Timestamp(day.date, tz=TZ)

panels = [("heart_rate", "Pulzus (bpm)"), ("stress", "Stressz"), ("body_battery", "Body Battery"),
          ("respiration", "Légzés (/perc)")]
fig, axes = plt.subplots(len(panels), 1, figsize=(14, 10), sharex=True)
for ax, (metric, label) in zip(axes, panels):
    m = day.metric(metric)
    if metric == "stress" and not m.empty:
        cat = pd.cut(m["value"], [0, 25, 50, 75, 100], labels=STRESS_LABELS, include_lowest=True)
        ax.bar(m["time"], m["value"], width=pd.Timedelta(minutes=3), color=cat.map(STRESS_COLORS).astype(str))
    elif metric == "body_battery":
        ax.fill_between(m["time"], m["value"], color="#3fb56b", alpha=0.6)
        ax.set_ylim(0, 100)
    else:
        m = day.metric(metric, break_gaps="10min")
        ax.plot(m["time"], m["value"], lw=0.9)
    for _, sl in day.sleeps.iterrows():
        ax.axvspan(sl["start"], sl["end"], color="#1f3a93", alpha=0.12)
    for _, a in day.activities.iterrows():
        ax.axvspan(a["start"], a["end"], color="orange", alpha=0.3)
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
for _, a in day.activities.iterrows():
    axes[0].annotate(a["name"], (a["start"], axes[0].get_ylim()[1]), fontsize=8, va="top", rotation=90)
axes[-1].set_xlim(t0, t0 + pd.Timedelta(days=1))
axes[-1].xaxis.set_major_locator(mdates.HourLocator(interval=2, tz=TZ))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=TZ))
fig.suptitle(f"{day.date}")
plt.tight_layout()

## Stressz eloszlás és edzéskészség tényezők
Garmin kategóriák: 0–25 pihenés, 26–50 alacsony, 51–75 közepes, 76–100 magas. Az edzéskészség tényezők 100% = legjobb.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
mins = day.stress_minutes()
ax1.barh(mins.index, mins.values / 60, color=[STRESS_COLORS[c] for c in mins.index])
ax1.set_xlabel("óra")
ax1.set_title("Stressz kategóriák (mért idő)")
ax1.invert_yaxis()

factors = {"sleep_pct": "Alvás (előző éj)", "sleep_history_pct": "Alvás előzmény", "recovery_time_pct": "Regeneráció",
           "acwr_pct": "Terhelés arány (ACWR)", "stress_history_pct": "Stressz előzmény", "hrv_pct": "HRV"}
if not morning.empty:
    r = morning.iloc[0]
    vals = pd.Series({v: r[k] for k, v in factors.items()}).dropna()
    ax2.barh(vals.index, vals.values, color="#4a90d9")
    ax2.set_xlim(0, 100)
    ax2.set_title(f"Reggeli edzéskészség: {r['score']} ({r['level']}, {r['feedback']})")
    ax2.invert_yaxis()
else:
    ax2.set_title("Nincs reggeli edzéskészség mérés")
plt.tight_layout()

day.readiness[["timestamp_local", "score", "level", "feedback", "recovery_time_min", "acute_load", "is_morning"]]

## Trendek (összes nap)
- **Nyugalmi pulzus ↓ / HRV ↑** általában jobb regenerációt jelez; érdemes a saját átlagodhoz viszonyítani.
- **Akut vs. krónikus terhelés:** a sáv a Garmin által optimálisnak tartott krónikus tartomány; ACWR ≈ 0,8–1,3 a kiegyensúlyozott, felette gyors terhelésnövekedés, alatta pihenés / visszaesés.

In [ ]:
tr = daily_trends(DB)
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

ax = axes[0]
ax.plot(tr["date"], tr["resting_hr"], "o-", color="#d9534f", label="nyugalmi pulzus")
ax.set_ylabel("bpm")
ax2 = ax.twinx()
ax2.plot(tr["date"], tr["hrv"], "o-", color="#4a90d9", label="HRV (éjszakai)")
ax2.plot(tr["date"], tr["hrv_weekly"], "--", color="#4a90d9", alpha=0.6, label="HRV (heti átlag)")
ax2.set_ylabel("ms")
ax.legend(loc="upper left"); ax2.legend(loc="upper right")

ax = axes[1]
ax.bar(tr["date"], tr["sleep_h"], color="#1f3a93", alpha=0.5, label="alvás (óra)")
ax.set_ylabel("óra")
ax2 = ax.twinx()
ax2.plot(tr["date"], tr["sleep_score"], "o-", color="#1f3a93", label="alvás pontszám")
ax2.set_ylim(0, 100)
ax.legend(loc="upper left"); ax2.legend(loc="upper right")

ax = axes[2]
ax.plot(tr["date"], tr["readiness_morning"], "o-", color="#3fb56b", label="reggeli edzéskészség")
ax.fill_between(tr["date"], tr["body_battery_low"], tr["body_battery_high"], color="#3fb56b", alpha=0.15,
                label="Body Battery min–max")
ax.set_ylim(0, 100)
ax.legend(loc="upper left")

ax = axes[3]
ax.fill_between(tr["date"], tr["chronic_load_min"], tr["chronic_load_max"], color="gray", alpha=0.2,
                label="optimális tartomány")
ax.plot(tr["date"], tr["chronic_load"], "-", color="gray", label="krónikus terhelés")
ax.plot(tr["date"], tr["acute_load"], "o-", color="#f08c3a", label="akut terhelés")
ax.set_ylabel("terhelés")
ax2 = ax.twinx()
ax2.plot(tr["date"], tr["acwr"], ":", color="black", label="ACWR")
ax2.axhspan(0.8, 1.3, color="green", alpha=0.05)
ax2.set_ylabel("ACWR")
ax.legend(loc="upper left"); ax2.legend(loc="upper right")

for ax in axes:
    ax.grid(alpha=0.3)
    ax.axvline(pd.Timestamp(day.date), color="red", lw=1, alpha=0.5)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.tight_layout()

tr.set_index("date").tail(7)